# Phase 1 Baseline: The Rigorous Strawman

This notebook is intentionally thin: reusable logic lives in `src/margin_of_error`. It reads the Phase 1 artifacts produced by `make train` and renders the residual analysis.

The Kaggle training split is random, not temporal; these metrics do not test regime robustness.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

METRIC_CARD = Path("../reports/phase1_metric_card.json")
RESIDUALS = Path("../reports/phase1_oof_residuals.csv")
FIGURE = Path("../reports/figures/01_price_error_vs_home_price.svg")

card = json.loads(METRIC_CARD.read_text())
residuals = pd.read_csv(RESIDUALS)

## Model Comparison

All rows are repeated 5-fold CV summaries. Dollar metrics use Duan smearing for log retransformation.

In [ ]:
models = pd.DataFrame(card["models"]).T
cols = [
    "rmse_log_mean",
    "rmse_log_std",
    "mae_dollars_mean",
    "mae_dollars_std",
    "rmse_dollars_mean",
    "rmse_dollars_std",
]
models[cols]

## Residual Spread

The primary Phase 1 strawman is the tuned LightGBM point model. It is deliberately strong enough to be taken seriously, then translated into dollars so its underwriting limitation is visible.

In [ ]:
pd.Series(card["residual_diagnostics"]["spread"])

## Signature Figure

Dollar prediction error is plotted against actual sale price. The orange band is a plausible $10K-$20K net flip-margin range, included only as a Phase 2 hypothesis prompt, not as a completed economics model.

In [ ]:
display(SVG(filename=str(FIGURE)))

## Phase 1 Framing Sentence

Hypothesis to be tested in Phase 2: A typical fix-and-flip net margin is on the order of $10-20K; this model's typical dollar error is $9,413. If that error is comparable to or larger than that margin, point predictions cannot safely underwrite a flip.

## Where The Model Fails

These diagnostics seed Phase 2: uncertainty is not uniform across homes.

In [ ]:
residuals.assign(price_bin=pd.qcut(residuals["SalePrice"], 5)).groupby("price_bin", observed=True)[
    "abs_error_dollars"
].agg(["count", "median", "mean", "max"])

In [ ]:
residuals.groupby("Neighborhood")["abs_error_dollars"].agg(["count", "median", "mean"]).sort_values(
    "median", ascending=False
).head(10)